# Validate the Silver project foundation

## Purpose

Prove that `silver_dpwh_project_component` and `silver_project` keep every Bronze DPWH row, never double count a budget, keep stable keys and lineage, and show every source-quality finding. The checks cover all seven data-quality attributes.

## Inputs

- `01-bronze.dpwh_projects` row count, `01-bronze.load_log`, and `04-validation.dq_results` for the gate.
- `02-silver.silver_dpwh_project_component` and `02-silver.silver_project`.

## Outputs

- One row per check in `04-validation.silver_dq_results`.

## How results work

- `stop` checks protect pipeline correctness. A failure blocks the next notebooks.
- `flag` checks keep source limits visible. They never block.
- Every check shows its failed rows and its denominator, `total_rows`.
- Evidence is saved before the gate runs, so a failed run still leaves a record.

## Not done here

This notebook reports. It never fixes, filters, or deletes rows.

## 1. Read the latest safe Bronze load

**What:** select the latest terminal DPWH load event.

**Why:** the Timeliness checks compare the Silver snapshot with this load.

**Protects:** Timeliness.

**Expected result:** one load row.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

CREATE OR REPLACE TEMPORARY VIEW latest_dpwh_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY table_name
            ORDER BY
                COALESCE(completed_at, started_at) DESC,
                started_at DESC
        ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'dpwh_projects'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

SELECT
    table_name,
    status,
    snapshot_id,
    rows_loaded
FROM latest_dpwh_load;

## 2. Calculate every metric in one pass per table

**What:** scan Bronze once for its count, the component table once, and the project table once. Each scan computes all of its metrics.

**Why:** grouped `COUNT_IF` expressions keep the cost to three scans, however many checks we add. The earlier notebook scanned the same tables many times and repeated two cells.

**Protects:** low cost and one place for every metric.

**Expected result:** one metric row.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_project_validation_metrics AS
WITH bronze_metrics AS (
    SELECT COUNT(*) AS bronze_rows
    FROM `01-bronze`.dpwh_projects
),

load_metrics AS (
    SELECT MAX(snapshot_id) AS bronze_snapshot_id
    FROM latest_dpwh_load
),

component_metrics AS (
    SELECT
        COUNT(*) AS component_rows,
        COUNT(*) - COUNT(DISTINCT component_key) AS duplicate_component_keys,
        COUNT_IF(source_row_ordinal > 1) AS repeated_identical_rows,
        COUNT_IF(contract_id IS NULL OR TRIM(contract_id) = '') AS missing_contract_ids,
        COUNT_IF(
            component_key IS NULL
            OR source_row_hash IS NULL
            OR source_row_ordinal IS NULL
            OR run_id IS NULL
            OR transformation_rule_version IS NULL
            OR taxonomy_version IS NULL
            OR _source_snapshot_id IS NULL
            OR TRIM(_source_snapshot_id) = ''
            OR _ingest_run_id IS NULL
            OR TRIM(_ingest_run_id) = ''
        ) AS missing_component_lineage,
        COUNT_IF(
            run_id <> SHA2(
                CONCAT_WS(
                    '||',
                    _source_snapshot_id,
                    transformation_rule_version,
                    taxonomy_version,
                    COALESCE(active_status_mapping_version, 'NO_APPROVED_STATUS_MAPPING')
                ),
                256
            )
        ) AS non_reproducible_component_runs,
        COUNT(DISTINCT _source_snapshot_id) AS component_snapshot_count,
        MAX(_source_snapshot_id) AS component_snapshot_id,
        COUNT(DISTINCT run_id) AS component_run_count,
        MAX(run_id) AS component_run_id,
        COUNT_IF(
            budget_quality_status IS NULL
            OR budget_quality_status NOT IN ('VALID', 'MISSING', 'UNPARSEABLE', 'NEGATIVE')
            OR progress_quality_status IS NULL
            OR progress_quality_status NOT IN ('VALID', 'MISSING', 'UNPARSEABLE', 'OUT_OF_RANGE')
            OR infra_year_quality_status IS NULL
            OR infra_year_quality_status NOT IN ('VALID', 'MISSING', 'UNPARSEABLE')
            OR category_mapping_state IS NULL
            OR category_mapping_state NOT IN ('MAPPED', 'UNMAPPED')
            OR category_classification_status IS NULL
            OR category_classification_status NOT IN (
                'MISSING_SOURCE_CATEGORY',
                'UNAPPROVED_SOURCE_CATEGORY',
                'MAPPED_SINGLE',
                'MAPPED_MULTI'
            )
            OR status_mapping_state IS NULL
            OR status_mapping_state NOT IN ('MAPPED', 'UNMAPPED')
        ) AS invalid_controlled_states,
        COUNT_IF(
            (progress_quality_status = 'VALID' AND NOT physical_progress_pct BETWEEN 0 AND 100)
            OR (progress_quality_status = 'OUT_OF_RANGE' AND physical_progress_pct BETWEEN 0 AND 100)
            OR (budget_quality_status = 'VALID' AND reported_budget_pesos < 0)
            OR (budget_quality_status = 'NEGATIVE' AND reported_budget_pesos >= 0)
            OR (budget_quality_status IN ('MISSING', 'UNPARSEABLE') AND reported_budget_pesos IS NOT NULL)
        ) AS inconsistent_quality_states,
        COUNT_IF(
            source_infra_types <> ARRAY_SORT(ARRAY_DISTINCT(source_infra_types))
            OR (
                category_mapping_state = 'MAPPED'
                AND (
                    SIZE(source_infra_types) = 0
                    OR SIZE(unapproved_source_infra_types) > 0
                    OR standardized_sector IS NULL
                    OR is_dpwh_flood_related IS NULL
                )
            )
            OR (category_mapping_state = 'UNMAPPED' AND standardized_sector IS NOT NULL)
            OR (category_classification_status = 'MAPPED_SINGLE' AND SIZE(source_infra_types) <> 1)
            OR (category_classification_status = 'MAPPED_MULTI' AND SIZE(source_infra_types) <= 1)
            OR (category_classification_status = 'MISSING_SOURCE_CATEGORY' AND SIZE(source_infra_types) <> 0)
            OR (category_classification_status = 'UNAPPROVED_SOURCE_CATEGORY' AND SIZE(unapproved_source_infra_types) = 0)
        ) AS invalid_component_category_resolutions,
        COUNT_IF(
            (category_mapping_state = 'MAPPED' AND taxonomy_version IS NULL)
            OR (status_mapping_state = 'MAPPED' AND status_mapping_version IS NULL)
        ) AS mapped_rows_without_version,
        COUNT_IF(component_categories_extraction_status <> 'EXTRACTED') AS category_extraction_finding_rows,
        COUNT_IF(category_mapping_state = 'MAPPED') AS mapped_category_rows,
        COUNT_IF(category_mapping_state = 'UNMAPPED') AS unmapped_category_rows,
        COUNT_IF(category_classification_status = 'MAPPED_SINGLE') AS single_category_component_rows,
        COUNT_IF(category_classification_status = 'MAPPED_MULTI') AS multi_category_component_rows,
        COUNT_IF(category_classification_status = 'MISSING_SOURCE_CATEGORY') AS missing_category_component_rows,
        COUNT_IF(category_classification_status = 'UNAPPROVED_SOURCE_CATEGORY') AS unapproved_category_component_rows,
        COUNT_IF(status_mapping_state = 'UNMAPPED') AS unmapped_status_rows,
        COUNT_IF(coordinate_status <> 'VALID_PAIR') AS coordinate_exception_rows,
        COUNT_IF(budget_quality_status = 'UNPARSEABLE') AS budget_unparseable_rows,
        COUNT_IF(budget_quality_status = 'NEGATIVE') AS negative_budget_rows,
        COUNT_IF(progress_quality_status = 'UNPARSEABLE') AS progress_unparseable_rows,
        COUNT_IF(progress_quality_status = 'OUT_OF_RANGE') AS progress_out_of_range_rows,
        COUNT_IF(infra_year_quality_status = 'UNPARSEABLE') AS infra_year_unparseable_rows,
        COUNT_IF(
            (start_date_raw IS NOT NULL AND start_date_parsed IS NULL)
            OR (completion_date_raw IS NOT NULL AND completion_date_parsed IS NULL)
        ) AS date_parse_failure_rows
    FROM `02-silver`.silver_dpwh_project_component
),

component_budget_control AS (
    SELECT COALESCE(SUM(reported_budget_pesos), CAST(0 AS DECIMAL(38, 2))) AS expected_project_budget
    FROM (
        SELECT
            contract_id,
            CASE WHEN COUNT(DISTINCT reported_budget_pesos) = 1 THEN MAX(reported_budget_pesos) END AS reported_budget_pesos
        FROM `02-silver`.silver_dpwh_project_component
        GROUP BY contract_id
    ) AS resolved_component_budgets
),

project_metrics AS (
    SELECT
        COUNT(*) AS project_rows,
        COALESCE(SUM(component_count), 0) AS accounted_component_rows,
        COUNT(*) - COUNT(DISTINCT contract_id) AS duplicate_contract_ids,
        COUNT(*) - COUNT(DISTINCT project_key) AS duplicate_project_keys,
        COUNT_IF(
            (distinct_budget_value_count = 1 AND reported_budget_pesos IS NULL)
            OR (distinct_budget_value_count > 1 AND reported_budget_pesos IS NOT NULL)
            OR (budget_resolution_status = 'CONFLICT' AND reported_budget_pesos IS NOT NULL)
        ) AS invalid_budget_resolutions,
        COUNT_IF(
            (progress_resolution_status = 'RESOLVED' AND zero_progress_flag IS NULL)
            OR (progress_resolution_status <> 'RESOLVED' AND zero_progress_flag IS NOT NULL)
            OR (zero_progress_flag AND physical_progress_pct <> 0)
            OR (NOT zero_progress_flag AND physical_progress_pct = 0)
        ) AS invalid_zero_progress_flags,
        COUNT_IF(
            source_infra_types <> ARRAY_SORT(ARRAY_DISTINCT(source_infra_types))
            OR (
                category_mapping_state = 'MAPPED'
                AND (
                    SIZE(source_infra_types) = 0
                    OR SIZE(unapproved_source_infra_types) > 0
                    OR standardized_sector IS NULL
                    OR is_dpwh_flood_related IS NULL
                )
            )
            OR (category_mapping_state = 'UNMAPPED' AND standardized_sector IS NOT NULL)
            OR (category_classification_status = 'MAPPED_SINGLE' AND SIZE(source_infra_types) <> 1)
            OR (category_classification_status = 'MAPPED_MULTI' AND SIZE(source_infra_types) <= 1)
            OR (category_classification_status = 'MISSING_SOURCE_CATEGORY' AND SIZE(source_infra_types) <> 0)
            OR (category_classification_status = 'UNAPPROVED_SOURCE_CATEGORY' AND SIZE(unapproved_source_infra_types) = 0)
        ) AS invalid_project_category_resolutions,
        COUNT_IF(
            description_resolution_status = 'CONFLICT'
            OR category_resolution_status = 'CONFLICT'
            OR status_resolution_status = 'CONFLICT'
            OR progress_resolution_status = 'CONFLICT'
            OR start_date_resolution_status = 'CONFLICT'
            OR completion_date_resolution_status = 'CONFLICT'
            OR reported_region_resolution_status = 'CONFLICT'
            OR budget_resolution_status = 'CONFLICT'
        ) AS project_conflict_rows,
        COUNT_IF(
            project_key IS NULL
            OR run_id IS NULL
            OR transformation_rule_version IS NULL
            OR source_snapshot_id IS NULL
            OR source_ingest_run_id IS NULL
            OR component_run_id IS NULL
            OR taxonomy_version IS NULL
        ) AS missing_project_lineage,
        COUNT_IF(
            run_id <> SHA2(
                CONCAT_WS(
                    '||',
                    COALESCE(source_snapshot_id, ''),
                    COALESCE(component_run_id, ''),
                    COALESCE(taxonomy_version, ''),
                    transformation_rule_version
                ),
                256
            )
        ) AS non_reproducible_project_runs,
        COUNT(DISTINCT source_snapshot_id) AS project_snapshot_count,
        MAX(source_snapshot_id) AS project_snapshot_id,
        COUNT(DISTINCT run_id) AS project_run_count,
        MAX(run_id) AS project_run_id,
        COUNT(DISTINCT component_run_id) AS project_component_run_count,
        MAX(component_run_id) AS project_component_run_id,
        COUNT(DISTINCT taxonomy_version) AS project_taxonomy_version_count,
        COUNT_IF(category_mapping_state = 'MAPPED') AS mapped_projects,
        COUNT_IF(category_mapping_state = 'UNMAPPED') AS unmapped_projects,
        COUNT_IF(category_classification_status = 'MAPPED_SINGLE') AS single_sector_projects,
        COUNT_IF(category_classification_status = 'MAPPED_MULTI') AS multi_sector_projects,
        COUNT_IF(category_classification_status = 'MISSING_SOURCE_CATEGORY') AS missing_category_projects,
        COUNT_IF(category_classification_status = 'UNAPPROVED_SOURCE_CATEGORY') AS unapproved_category_projects,
        COUNT_IF(is_dpwh_flood_related) AS dpwh_flood_related_projects,
        COALESCE(SUM(reported_budget_pesos), CAST(0 AS DECIMAL(38, 2))) AS total_resolved_reported_budget,
        COALESCE(SUM(CASE WHEN category_mapping_state = 'MAPPED' THEN reported_budget_pesos ELSE CAST(0 AS DECIMAL(18, 2)) END), CAST(0 AS DECIMAL(38, 2))) AS mapped_reported_budget,
        COALESCE(SUM(CASE WHEN category_mapping_state = 'UNMAPPED' THEN reported_budget_pesos ELSE CAST(0 AS DECIMAL(18, 2)) END), CAST(0 AS DECIMAL(38, 2))) AS unmapped_reported_budget,
        COALESCE(SUM(CASE WHEN is_dpwh_flood_related THEN reported_budget_pesos ELSE CAST(0 AS DECIMAL(18, 2)) END), CAST(0 AS DECIMAL(38, 2))) AS dpwh_flood_related_reported_budget,
        MAX(transformation_rule_version) AS project_rule_version
    FROM `02-silver`.silver_project
)

SELECT *
FROM bronze_metrics
CROSS JOIN load_metrics
CROSS JOIN component_metrics
CROSS JOIN component_budget_control
CROSS JOIN project_metrics;

SELECT * FROM silver_project_validation_metrics;

## 3. Express each check as one row

**What:** turn each metric into a check with its attribute, action, failed rows, denominator, and details. One final query sets the status the same way for every check.

**Why:** one status rule for all checks means no check can be scored differently by mistake. Adding a check is one `UNION ALL` line.

**Protects:** all seven attributes.

| Attribute | What the checks prove |
| --- | --- |
| Completeness | Every Bronze row is a component. Every component is counted in a project. Unmapped and coordinate gaps are counted. |
| Uniqueness | Component keys, Contract IDs, and project keys are unique. Repeated identical source rows are counted, not removed. |
| Validity | Contract IDs exist. Quality and mapping states use the controlled values. Unparseable, negative, and out-of-range values are counted. |
| Accuracy | Budgets follow the no-double-count rule. Quality statuses agree with the values. The zero-progress flag agrees with progress. |
| Consistency | Mapped rows carry their mapping version. Project conflicts stay visible. |
| Auditability | Keys, run IDs, rule versions, and snapshot lineage exist and reproduce. |
| Timeliness | Silver holds one snapshot, the same one as the latest Bronze load. The project run uses the current component run. |

**Expected result:** every `stop` check is `PASS`. `flag` checks may be `FLAG`.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_project_checks AS
WITH m AS (
    SELECT * FROM silver_project_validation_metrics
),

checks AS (
    SELECT
        'silver_dpwh_project_component' AS table_name,
        'Completeness' AS attribute,
        'Component table contains rows' AS check_name,
        CASE WHEN component_rows = 0 THEN 1 ELSE 0 END AS failed_rows,
        1 AS total_rows,
        'stop' AS action,
        CONCAT('Component rows=', component_rows) AS details
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Completeness',
        'Bronze DPWH rows reconcile to component rows',
        ABS(component_rows - bronze_rows), bronze_rows, 'stop',
        CONCAT('Bronze rows=', bronze_rows, ', component rows=', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Completeness',
        'Project component counts reconcile to component rows',
        ABS(accounted_component_rows - component_rows), component_rows, 'stop',
        CONCAT('Component rows=', component_rows, ', accounted rows=', accounted_component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Completeness',
        'Component category extraction findings remain visible',
        category_extraction_finding_rows, component_rows, 'flag',
        CONCAT('Rows missing source JSON or usable componentCategories=', category_extraction_finding_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Completeness',
        'Unmapped categories remain visible for taxonomy review',
        unmapped_category_rows, component_rows, 'flag',
        CONCAT('Rows with no approved category rule=', unmapped_category_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Completeness',
        'Unexpected component category tokens remain visible',
        unapproved_category_component_rows, component_rows, 'flag',
        CONCAT('Rows with unapproved component category tokens=', unapproved_category_component_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Completeness',
        'Project taxonomy coverage is reported',
        unmapped_projects, project_rows, 'flag',
        CONCAT(
            'Mapped projects=', mapped_projects,
            ', unmapped projects=', unmapped_projects,
            ', mapped project percentage=', ROUND(100.0 * mapped_projects / NULLIF(project_rows, 0), 4),
            ', mapped budget=', mapped_reported_budget,
            ', unmapped budget=', unmapped_reported_budget,
            ', mapped budget percentage=', ROUND(100.0 * mapped_reported_budget / NULLIF(total_resolved_reported_budget, 0), 4)
        )
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Completeness',
        'Unmapped statuses remain visible for status review',
        unmapped_status_rows, component_rows, 'flag',
        CONCAT('Rows with no approved status rule=', unmapped_status_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Completeness',
        'Coordinate exceptions remain visible',
        coordinate_exception_rows, component_rows, 'flag',
        CONCAT('Missing, partial, or unparseable pairs=', coordinate_exception_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Uniqueness',
        'Component keys are unique',
        duplicate_component_keys, component_rows, 'stop',
        CONCAT('Duplicate component keys=', duplicate_component_keys)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Uniqueness',
        'Repeated identical source rows are kept and counted',
        repeated_identical_rows, component_rows, 'flag',
        CONCAT('Rows identical to an earlier row on every Silver field=', repeated_identical_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Uniqueness',
        'Project Contract ID grain is unique',
        duplicate_contract_ids, project_rows, 'stop',
        CONCAT('Duplicate Contract IDs=', duplicate_contract_ids)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Uniqueness',
        'Project keys are unique',
        duplicate_project_keys, project_rows, 'stop',
        CONCAT('Duplicate project keys=', duplicate_project_keys)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Contract ID is present before project consolidation',
        missing_contract_ids, component_rows, 'stop',
        CONCAT('Missing or blank Contract IDs=', missing_contract_ids)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Quality and mapping states use the controlled values',
        invalid_controlled_states, component_rows, 'stop',
        CONCAT('Rows with an uncontrolled state=', invalid_controlled_states)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Unparseable budgets remain visible',
        budget_unparseable_rows, component_rows, 'flag',
        CONCAT('Unparseable budget rows=', budget_unparseable_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Negative budgets remain visible',
        negative_budget_rows, component_rows, 'flag',
        CONCAT('Negative budget rows=', negative_budget_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Unparseable progress values remain visible',
        progress_unparseable_rows, component_rows, 'flag',
        CONCAT('Unparseable progress rows=', progress_unparseable_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Progress outside 0 to 100 remains visible',
        progress_out_of_range_rows, component_rows, 'flag',
        CONCAT('Out-of-range progress rows=', progress_out_of_range_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Unparseable infrastructure years remain visible',
        infra_year_unparseable_rows, component_rows, 'flag',
        CONCAT('Unparseable infra_year rows=', infra_year_unparseable_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Validity',
        'Unparseable dates remain visible',
        date_parse_failure_rows, component_rows, 'flag',
        CONCAT('Rows with an unparseable start or completion date=', date_parse_failure_rows, ' of ', component_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Accuracy',
        'Project budgets follow the no-double-count resolution rule',
        invalid_budget_resolutions, project_rows, 'stop',
        CONCAT('Invalid budget resolutions=', invalid_budget_resolutions)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Accuracy',
        'Quality statuses agree with the parsed values',
        inconsistent_quality_states, component_rows, 'stop',
        CONCAT('Rows whose status disagrees with the value=', inconsistent_quality_states)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Accuracy',
        'Zero-progress flag agrees with resolved progress',
        invalid_zero_progress_flags, project_rows, 'stop',
        CONCAT('Invalid zero-progress flags=', invalid_zero_progress_flags)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Consistency',
        'Mapped rows carry their mapping version',
        mapped_rows_without_version, component_rows, 'stop',
        CONCAT('Mapped rows without a version=', mapped_rows_without_version)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Consistency',
        'Component category arrays and classifications are internally consistent',
        invalid_component_category_resolutions, component_rows, 'stop',
        CONCAT('Invalid component category resolutions=', invalid_component_category_resolutions)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Consistency',
        'Project category arrays and classifications are internally consistent',
        invalid_project_category_resolutions, project_rows, 'stop',
        CONCAT('Invalid project category resolutions=', invalid_project_category_resolutions)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Consistency',
        'Projects use one taxonomy version',
        CASE WHEN project_taxonomy_version_count = 1 THEN 0 ELSE 1 END, 1, 'stop',
        CONCAT('Distinct project taxonomy versions=', project_taxonomy_version_count)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Consistency',
        'Project attribute conflicts remain visible',
        project_conflict_rows, project_rows, 'flag',
        CONCAT('Projects with at least one conflicting attribute=', project_conflict_rows, ' of ', project_rows)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Accuracy',
        'Project reported budget reconciles to the component resolution rule',
        CASE WHEN total_resolved_reported_budget = expected_project_budget THEN 0 ELSE 1 END, 1, 'stop',
        CONCAT(
            'Expected resolved budget=', expected_project_budget,
            ', published resolved budget=', total_resolved_reported_budget
        )
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Accuracy',
        'Mapped and unmapped budgets reconcile to total resolved budget',
        CASE WHEN mapped_reported_budget + unmapped_reported_budget = total_resolved_reported_budget THEN 0 ELSE 1 END, 1, 'stop',
        CONCAT(
            'Mapped budget=', mapped_reported_budget,
            ', unmapped budget=', unmapped_reported_budget,
            ', total resolved budget=', total_resolved_reported_budget
        )
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Auditability',
        'Component identity and source lineage are populated',
        missing_component_lineage, component_rows, 'stop',
        CONCAT('Rows missing identity or lineage=', missing_component_lineage)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Auditability',
        'Component run IDs reproduce from snapshot, taxonomy, status config, and rule version',
        non_reproducible_component_runs, component_rows, 'stop',
        CONCAT('Non-reproducible component run IDs=', non_reproducible_component_runs)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Auditability',
        'Project keys, run IDs, and lineage are populated',
        missing_project_lineage, project_rows, 'stop',
        CONCAT('Projects missing key or lineage=', missing_project_lineage)
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Auditability',
        'Project run IDs reproduce from snapshot, component run, taxonomy, and rule version',
        non_reproducible_project_runs, project_rows, 'stop',
        CONCAT('Non-reproducible project run IDs=', non_reproducible_project_runs)
    FROM m
    UNION ALL
    SELECT
        'silver_dpwh_project_component', 'Timeliness',
        'Components hold one snapshot equal to the latest Bronze load',
        CASE
            WHEN component_snapshot_count = 1 AND component_snapshot_id = bronze_snapshot_id THEN 0
            ELSE 1
        END,
        1, 'stop',
        CONCAT(
            'Component snapshots=', component_snapshot_count,
            ', component snapshot=', COALESCE(component_snapshot_id, 'NONE'),
            ', Bronze snapshot=', COALESCE(bronze_snapshot_id, 'NONE')
        )
    FROM m
    UNION ALL
    SELECT
        'silver_project', 'Timeliness',
        'Projects come from the current component run and snapshot',
        CASE
            WHEN project_run_count = 1
                AND project_snapshot_count = 1
                AND project_component_run_count = 1
                AND project_snapshot_id = component_snapshot_id
                AND project_component_run_id = component_run_id
                AND component_run_count = 1
                THEN 0
            ELSE 1
        END,
        1, 'stop',
        CONCAT(
            'Project runs=', project_run_count,
            ', project snapshots=', project_snapshot_count,
            ', component runs seen by projects=', project_component_run_count,
            ', component runs=', component_run_count
        )
    FROM m
),

context AS (
    SELECT
        COALESCE(project_run_id, 'NO_PROJECT_RUN') AS pipeline_run_id,
        component_snapshot_id AS source_snapshot_id,
        CONCAT(COALESCE(project_rule_version, 'NO_PROJECT_RULE'), '+silver_project_validation_v3') AS rule_version,
        SHA2(
            CONCAT_WS(
                '|',
                COALESCE(project_run_id, ''),
                COALESCE(component_run_id, ''),
                'silver_project_validation_v3'
            ),
            256
        ) AS validation_run_id,
        CURRENT_TIMESTAMP() AS run_ts
    FROM m
)

SELECT
    context.validation_run_id,
    context.pipeline_run_id,
    'silver' AS layer,
    checks.table_name,
    context.source_snapshot_id,
    context.rule_version,
    checks.attribute AS data_quality_attribute,
    checks.check_name AS data_quality_check,
    CAST(checks.failed_rows AS BIGINT) AS failed_rows,
    CAST(checks.total_rows AS BIGINT) AS total_rows,
    CASE
        WHEN checks.total_rows > 0 THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4)
    END AS percentage,
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action,
    checks.details,
    context.run_ts
FROM checks
CROSS JOIN context;

## 4. Save the evidence, then enforce the gate

**What:** merge the checks into `04-validation.silver_dq_results`, then stop the run if any `stop` check failed.

**Why:** the validation run ID hashes the project run, the component run, and the validation rule version. An exact rerun updates the same rows instead of adding copies. The merge runs before the assert, so a failed run still leaves evidence.

**Protects:** Auditability and safe reruns.

**Expected result:** the assert passes. The final query lists `FAIL` first, then `FLAG`, then `PASS`.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL,
    pipeline_run_id STRING NOT NULL,
    layer STRING NOT NULL,
    table_name STRING NOT NULL,
    source_snapshot_id STRING,
    rule_version STRING,
    data_quality_attribute STRING NOT NULL,
    data_quality_check STRING NOT NULL,
    failed_rows BIGINT,
    total_rows BIGINT,
    percentage DOUBLE,
    status STRING NOT NULL,
    action STRING NOT NULL,
    details STRING,
    run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING silver_project_checks AS source
    ON target.validation_run_id = source.validation_run_id
    AND target.table_name = source.table_name
    AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Silver project checks failed'
    )
)
FROM silver_project_checks;

# Summary

- Validated the Silver project foundation with checks across all seven data-quality attributes.
- `stop` checks prove row preservation, unique keys, deterministic category arrays, the no-double-count budget rule, controlled states, reproducible run IDs, lineage, and one current snapshot.
- `flag` checks count missing or unapproved component categories, unmapped statuses, coordinate gaps, repeated identical source rows, and unparseable, negative, or out-of-range values. Each count shows its denominator.
- The coverage result reports mapped and unmapped projects and reported budgets, single-sector and multi-sector counts, missing categories, and DPWH flood-related projects.
- Nothing is fixed or deleted here. Findings go to `04-validation.silver_dq_results`.
- Next: rerun `02_silver/08_silver_project_region_map` and `02_silver/10_silver_project_flood_map`, then their validators, because the project `run_id` changed.

The query below lists every check, failures first.

In [ ]:
%sql
SELECT
    data_quality_attribute,
    status,
    action,
    failed_rows,
    total_rows,
    percentage,
    table_name,
    data_quality_check,
    details
FROM silver_project_checks
ORDER BY
    CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END,
    data_quality_attribute,
    data_quality_check;

SELECT
    project_rows AS total_projects,
    mapped_projects,
    unmapped_projects,
    ROUND(100.0 * mapped_projects / NULLIF(project_rows, 0), 4) AS mapped_project_percentage,
    total_resolved_reported_budget,
    mapped_reported_budget,
    unmapped_reported_budget,
    ROUND(100.0 * mapped_reported_budget / NULLIF(total_resolved_reported_budget, 0), 4) AS mapped_budget_percentage,
    single_sector_projects,
    multi_sector_projects,
    missing_category_projects,
    unapproved_category_projects,
    dpwh_flood_related_projects,
    dpwh_flood_related_reported_budget
FROM silver_project_validation_metrics;